# 03. Insights, Validación y Enfoque de Producto

## Qué quiero lograr

En este cuaderno quiero cerrar la distancia entre análisis y producto. Mi objetivo ya no es limpiar ni perfilar, sino decidir qué hallazgos vale la pena mostrar, qué preguntas sí puedo contestar y cómo traducir eso a un backend grounded y a una demo defendible.

## Cómo voy a filtrar los resultados

Voy a distinguir entre:

1. observaciones con evidencia suficiente,
2. interpretaciones razonables pero acotadas,
3. claims que todavía no debo prometer.

Ese filtro, para mí, es una parte central del trabajo de AI engineering.


## Preparación del entorno

Voy a trabajar directamente sobre `data/processed/` porque eso se parece mucho más a cómo operará el producto real. No quiero que dashboard o bot dependan de releer el raw a cada consulta.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROCESSED_DIR = ROOT / "data" / "processed"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 120)


## Helpers para traducir análisis a producto

Defino helpers pequeños para resumir cobertura, confianza y answerability. La razón es muy concreta: quiero que las reglas que uso para decidir si algo entra al dashboard o al bot sean visibles y repetibles.


In [ ]:
def coverage_flag(ratio: float | int | None) -> str:
    # Resume cobertura observada en una etiqueta compacta.
    if ratio is None or pd.isna(ratio):
        return "sin evidencia"
    if ratio >= 0.95:
        return "alta"
    if ratio >= 0.80:
        return "media"
    return 'baja'


def support_status(is_supported: bool, caution: bool = False) -> str:
    # Expresa si una pregunta o insight es apto para producto.
    if not is_supported:
        return "no soportado"
    if caution:
        return "soportado con cautela"
    return "soportado"


def hourly_coverage_ratio(n_points: float | int | None) -> float | None:
    # Normaliza una hora observada contra el máximo esperado de 360 puntos.
    if n_points is None or pd.isna(n_points):
        return None
    return min(float(n_points) / 360.0, 1.0)


## 1. Cargo la capa procesada y el reporte de calidad

Primero quiero traer todas las capas que voy a cruzar para tomar decisiones. Me importa especialmente cargar también el `quality_report`, porque no quiero interpretar señales sin el contexto de cobertura.


In [ ]:
canonical_df = pd.read_csv(
    PROCESSED_DIR / "availability_long_canonical.csv",
    parse_dates=["timestamp", "window_start", "window_end", "hour_bucket"],
)
hourly_df = pd.read_csv(PROCESSED_DIR / "availability_hourly.csv", parse_dates=["hour_bucket"])
daily_df = pd.read_csv(
    PROCESSED_DIR / "availability_daily.csv",
    parse_dates=["first_timestamp", "last_timestamp"],
)
anomaly_df = pd.read_csv(
    PROCESSED_DIR / "availability_hourly_anomalies.csv",
    parse_dates=["hour_bucket"],
)
step_change_df = pd.read_csv(
    PROCESSED_DIR / "availability_step_changes.csv",
    parse_dates=["timestamp"],
)
quality_report = json.loads((PROCESSED_DIR / "availability_quality_report.json").read_text())

processed_overview = pd.DataFrame(
    [
        {
            "observed_start": quality_report["observed_start"],
            "observed_end": quality_report["observed_end"],
            "canonical_timestamps": quality_report["canonical_timestamp_count"],
            "missing_ratio_full_range": quality_report["missing_ratio_full_range"],
            "duplicate_window_groups": quality_report["duplicate_window_groups"],
            "incomplete_window_records": quality_report["incomplete_window_records"],
        }
    ]
)

display(processed_overview)


Esta tabla me ayuda a anclar toda la discusión posterior. Si sé desde el inicio que la cobertura no es perfecta, entonces ya sé que cualquier lectura fuerte necesita un lenguaje prudente y trazable.


## 2. Quiero identificar qué historias diarias valen la pena contar

En esta sección no quiero solo mirar la tabla diaria; quiero convertirla en candidatos reales de narrativa. Voy a revisar días fuertes, días débiles, cambios respecto al día anterior y siempre voy a acompañarlo de cobertura.


In [ ]:
daily_story = daily_df.copy()
daily_story["coverage_flag"] = daily_story["coverage_ratio_in_observed_span"].map(coverage_flag)
daily_story["prev_mean_value"] = daily_story["mean_value"].shift(1)
daily_story["delta_vs_prev_day"] = daily_story["mean_value"] - daily_story["prev_mean_value"]
daily_story["pct_vs_prev_day"] = (
    daily_story["delta_vs_prev_day"]
    / daily_story["prev_mean_value"].replace(0, pd.NA)
    * 100
)
daily_story["story_ready"] = np.where(
    daily_story["coverage_ratio_in_observed_span"] >= 0.80,
    "sí",
    "con cautela",
)

best_days = daily_story.sort_values("mean_value", ascending=False)[
    ["date", "mean_value", "coverage_ratio_in_observed_span", "coverage_flag", "story_ready"]
].head(3)

weak_days = daily_story.sort_values("mean_value", ascending=True)[
    ["date", "mean_value", "coverage_ratio_in_observed_span", "coverage_flag", "story_ready"]
].head(3)

display(daily_story.round(3))
display(best_days.round(3))
display(weak_days.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
daily_story.plot(
    x="date",
    y="mean_value",
    kind="line",
    marker="o",
    ax=axes[0],
    title="Narrativa diaria: nivel medio de señal",
)
daily_story.plot(
    x="date",
    y="coverage_ratio_in_observed_span",
    kind="bar",
    ax=axes[1],
    title="Narrativa diaria: cobertura observada",
)
axes[0].set_xlabel("Día")
axes[0].set_ylabel("Valor medio")
axes[1].set_xlabel("Día")
axes[1].set_ylabel("Cobertura")
plt.xticks(rotation=45)
plt.tight_layout()


Aquí me interesa evitar una mala costumbre: contar el “mejor” o “peor” día sin decir con qué cobertura llegó. Para demo y producto, me parece mucho más serio mostrar el valor junto con la confianza o, al menos, con una advertencia contextual.


## 3. Quiero ver si el patrón intradiario justifica una visual principal

El dashboard y el bot ganan mucho valor si existe una forma diaria clara. Por eso acá construyo un perfil intradiario más explícito: quiero detectar horas pico, horas valle y estabilidad promedio por hora.


In [ ]:
hour_profile = (
    hourly_df.groupby("hour")
    .agg(
        mean_of_hourly_means=("mean_value", "mean"),
        median_of_hourly_means=("mean_value", "median"),
        std_of_hourly_means=("mean_value", "std"),
        avg_n_points=("n_points", "mean"),
    )
    .reset_index()
)
hour_profile["hourly_coverage_ratio"] = hour_profile["avg_n_points"].map(hourly_coverage_ratio)
hour_profile["coverage_flag"] = hour_profile["hourly_coverage_ratio"].map(coverage_flag)
hour_profile["rank_high"] = hour_profile["mean_of_hourly_means"].rank(ascending=False, method="dense")
hour_profile["rank_low"] = hour_profile["mean_of_hourly_means"].rank(ascending=True, method="dense")

peak_hours = hour_profile.sort_values("mean_of_hourly_means", ascending=False).head(5)
low_hours = hour_profile.sort_values("mean_of_hourly_means", ascending=True).head(5)

display(hour_profile.round(3))
display(peak_hours.round(3))
display(low_hours.round(3))

fig, ax = plt.subplots(figsize=(12, 4))
hour_profile.plot(
    x="hour",
    y=["mean_of_hourly_means", "median_of_hourly_means"],
    marker="o",
    ax=ax,
    title="Perfil intradiario agregado",
)
ax.set_xlabel("Hora del día")
ax.set_ylabel("Valor")
plt.tight_layout()


Este bloque es especialmente valioso para producto. Si la señal tiene un perfil intradiario estable, entonces ya no estoy construyendo una app que solo lista números: estoy construyendo una herramienta que entiende qué es normal a cada hora y qué se desvía de ese baseline.


## 4. Quiero revisar anomalías y saltos con un filtro de confianza

No toda desviación merece convertirse en highlight. Acá voy a añadir una lectura más prudente para decidir qué anomalías valen como insight y cuáles deberían quedar solo como contexto exploratorio o técnico.


In [ ]:
reviewed_anomalies = anomaly_df.copy()
reviewed_anomalies["hourly_coverage_ratio"] = reviewed_anomalies["n_points"].map(hourly_coverage_ratio)
reviewed_anomalies["coverage_flag"] = reviewed_anomalies["hourly_coverage_ratio"].map(coverage_flag)
reviewed_anomalies["story_support"] = np.where(
    (reviewed_anomalies["hourly_coverage_ratio"] >= 0.95)
    & (reviewed_anomalies["zscore_vs_hour_baseline"].abs() >= 1.5),
    "fuerte",
    "moderado",
)
reviewed_anomalies["demo_candidate"] = np.where(
    reviewed_anomalies["story_support"] == "fuerte",
    "sí",
    "con cautela",
)

reviewed_step_changes = step_change_df.copy()
reviewed_step_changes["technical_relevance"] = np.select(
    [
        (~reviewed_step_changes["is_gap_or_reset"]) & (reviewed_step_changes["delta_10s"].abs() >= 100000),
        ~reviewed_step_changes["is_gap_or_reset"],
    ],
    ["alta", "media"],
    default="baja",
)

display(reviewed_anomalies.round(3))
display(reviewed_step_changes.round(3))


Aquí me interesa transmitir una postura madura: una anomalía puede ser útil sin que eso implique que ya entiendo la causa. Lo que sí puedo hacer bien es detectar el desvío, cuantificarlo y mostrar con qué confianza apareció.


## 5. Quiero convertir el análisis en KPIs reales de dashboard

En lugar de una lista genérica de KPIs, aquí quiero proponer indicadores concretos y calcular un snapshot con los datos procesados. La idea es que el dashboard principal salga de aquí, no de intuiciones sueltas.


In [ ]:
dashboard_kpi_snapshot = pd.DataFrame(
    [
        {
            "metric": "Nivel medio de señal del período",
            "value": daily_story["mean_value"].mean(),
            "why_it_matters": "Resume el comportamiento agregado del período sin inventar granularidad por tienda.",
            "caution": "Debe mostrarse junto con cobertura del período.",
        },
        {
            "metric": "Cobertura efectiva del histórico observado",
            "value": 1 - quality_report["missing_ratio_full_range"],
            "why_it_matters": "Hace visible la salud del dato y evita sobreconfianza.",
            "caution": "No representa disponibilidad de negocio; representa continuidad observada del dataset.",
        },
        {
            "metric": "Hora típica de mayor señal",
            "value": int(hour_profile.sort_values("mean_of_hourly_means", ascending=False).iloc[0]["hour"]),
            "why_it_matters": "Ayuda a contar el patrón intradiario y a contextualizar comparaciones.",
            "caution": "Habla de patrón agregado, no de causalidad.",
        },
        {
            "metric": "Horas anómalas fuertes detectadas",
            "value": int((reviewed_anomalies["story_support"] == "fuerte").sum()),
            "why_it_matters": "Señala desvíos que merecen explicación o inspección adicional.",
            "caution": "Debe acompañarse de baseline y cobertura.",
        },
    ]
)

dashboard_sections = pd.DataFrame(
    [
        {
            "section": "Resumen ejecutivo",
            "what_to_show": "Nivel medio, cobertura y rango observado del período.",
            "source": "availability_daily.csv + quality_report",
        },
        {
            "section": "Patrón intradiario",
            "what_to_show": "Perfil horario típico y comparación contra baseline.",
            "source": "availability_hourly.csv",
        },
        {
            "section": "Anomalías con contexto",
            "what_to_show": "Horas desviadas con coverage flag y magnitud del z-score.",
            "source": "availability_hourly_anomalies.csv",
        },
        {
            "section": "Calidad del dato",
            "what_to_show": "Cobertura, duplicados tratados e incompletitud relevante.",
            "source": "quality_report + window metadata",
        },
    ]
)

display(dashboard_kpi_snapshot.round(3))
display(dashboard_sections)


Esta celda es importante porque obliga a pasar de “insights interesantes” a “producto defendible”. Si un KPI no puedo explicarlo con fuente, utilidad y cautela, entonces no debería entrar al dashboard todavía.


## 6. Quiero dejar explícito qué preguntas sí puede responder el bot

Acá quiero traducir la analítica a capacidades concretas del chatbot. Mi criterio es duro: el bot solo debería responder aquello que pueda resolverse con consultas determinísticas sobre `processed/` y luego formatearse con LLM.


In [ ]:
bot_capability_matrix = pd.DataFrame(
    [
        {
            "intent": "trend_summary",
            "example_question": "¿Cómo evolucionó la métrica entre el 8 y el 10 de febrero?",
            "support_status": support_status(True),
            "deterministic_source": "availability_daily.csv",
            "why": "Se resuelve con comparación temporal directa y cobertura por día.",
        },
        {
            "intent": "intraday_pattern",
            "example_question": "¿Qué horas suelen ser más altas o más bajas?",
            "support_status": support_status(True),
            "deterministic_source": "availability_hourly.csv",
            "why": "Existe un baseline por hora que permite responder con evidencia agregada.",
        },
        {
            "intent": "anomaly_review",
            "example_question": "¿Qué horas se desviaron más de su baseline?",
            "support_status": support_status(True, caution=True),
            "deterministic_source": "availability_hourly_anomalies.csv",
            "why": "Es respondible, pero siempre debería incluir cobertura/confianza.",
        },
        {
            "intent": "data_quality_status",
            "example_question": "¿Qué tan completo es el histórico y qué limitaciones tiene?",
            "support_status": support_status(True),
            "deterministic_source": "availability_quality_report.json",
            "why": "La calidad del dato está materializada de forma explícita.",
        },
        {
            "intent": "store_ranking",
            "example_question": "¿Qué tiendas estuvieron peor?",
            "support_status": support_status(False),
            "deterministic_source": "no aplica",
            "why": "El dataset no expone entidades de tienda ni granularidad individual.",
        },
        {
            "intent": "root_cause",
            "example_question": "¿Por qué cayó la métrica ese día?",
            "support_status": support_status(False),
            "deterministic_source": "no aplica",
            "why": "Puedo describir el cambio, pero no inferir causalidad con este dataset solo.",
        },
    ]
)

display(bot_capability_matrix)


Para mí esta matriz es de mucho valor porque demuestra autocontrol técnico. Un bot bueno no es el que inventa más; es el que responde bien dentro de un dominio claramente acotado y sabe cuándo no debe afirmar algo.


## 7. Quiero aterrizar qué consultas determinísticas debería exponer el backend

El bot grounded necesita una capa intermedia clara. En esta celda quiero convertir lo anterior en familias de consulta que luego se puedan mapear a endpoints o servicios backend.


In [ ]:
backend_query_contract = pd.DataFrame(
    [
        {
            "query_family": "overview_period",
            "response_shape": "nivel medio, cobertura, rango observado, advertencias",
            "primary_source": "availability_daily.csv + quality_report",
            "why": "Es la base del dashboard principal y de preguntas ejecutivas.",
        },
        {
            "query_family": "compare_periods",
            "response_shape": "diferencia absoluta/relativa entre dos rangos comparables",
            "primary_source": "availability_daily.csv o availability_hourly.csv",
            "why": "Permite comparación temporal grounded sin usar el LLM para calcular.",
        },
        {
            "query_family": "intraday_profile",
            "response_shape": "baseline por hora, hora pico, hora valle, horas con mejor cobertura",
            "primary_source": "availability_hourly.csv",
            "why": "Sostiene gráficos y respuestas semánticas sobre el comportamiento típico.",
        },
        {
            "query_family": "anomaly_context",
            "response_shape": "desviación, baseline, cobertura, nivel de confianza",
            "primary_source": "availability_hourly_anomalies.csv + availability_step_changes.csv",
            "why": "Permite responder sobre anomalías sin perder grounding.",
        },
        {
            "query_family": "quality_status",
            "response_shape": "cobertura, gaps, duplicados tratados, limitaciones del rango",
            "primary_source": "availability_quality_report.json + window metadata",
            "why": "Reduce alucinaciones y obliga a que el bot contextualice sus respuestas.",
        },
    ]
)

display(backend_query_contract)


## 8. Qué concluyo para producto y demo

Después de todo el recorrido, lo que considero defendible es esto:

- el producto correcto es analytics-first sobre una señal agregada,
- el dashboard debe priorizar tendencia, patrón intradiario, anomalías con contexto y calidad del dato,
- el bot debe trabajar como intérprete semántico de consultas determinísticas y no como generador de verdad numérica,
- la ausencia de granularidad por tienda no es una debilidad si la explico bien: es una restricción del dataset que estoy respetando con criterio.

Eso, bien contado, demuestra más madurez que prometer capacidades no soportadas por los datos.
